In [2]:
import pandas as pd
from statsmodels.tsa.ardl import ardl_select_order

df = pd.read_csv(
    "../data/processed/passthrough_dataset.csv",
    parse_dates=["date"]
).set_index("date")

df = df[["cpi_transport", "ron97", "diesel"]]
df = df.asfreq("ME")

df.head()

,cpi_transport,ron97,diesel
date,,,
2020-01-31,115.0,2.5725,2.1800
2020-02-29,113.9,2.3780,2.1340
2020-03-31,104.0,1.9275,1.8150
2020-04-30,90.0,1.5625,1.4675
2020-05-31,90.9,1.6240,1.4740


In [3]:
from statsmodels.tsa.stattools import adfuller

adf = adfuller(df["cpi_transport"])
print("ADF p-value (CPI Transport):", adf[1])


ADF p-value (CPI Transport): 2.6843711971042736e-07


In [4]:
sel = ardl_select_order(
    endog=df["cpi_transport"],
    exog=df[["ron97", "diesel"]],
    maxlag=4,
    maxorder=4,
    ic="aic",
    trend="c"
)

print(sel.model)


In [5]:
ardl_ct = sel.model.fit()
print(ardl_ct.summary())


                              ARDL Model Results                              
Dep. Variable:          cpi_transport   No. Observations:                   71
Model:                  ARDL(4, 1, 0)   Log Likelihood                 -58.406
Method:               Conditional MLE   S.D. of innovations              0.579
Date:                Thu, 08 Jan 2026   AIC                            134.813
Time:                        08:43:17   BIC                            154.655
Sample:                    05-31-2020   HQIC                           142.665
                         - 11-30-2025                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               12.0481      2.076      5.803      0.000       7.894      16.202
cpi_transport.L1     1.1592      0.059     19.571      0.000       1.041       1.278
cpi_transport.L2    -0.4720 

In [6]:
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.tsatools import lagmat
from statsmodels.stats.diagnostic import het_breuschpagan


In [8]:
res = ardl_ct

def bg_test_ardl_aligned(res, nlags=2):
    X0_full = np.asarray(res.model.data.exog)
    u = np.asarray(res.resid)

    n_u = len(u)
    if n_u <= nlags + 5:
        raise ValueError(f"Too few residual observations ({n_u}) for nlags={nlags}.")

    # Align exog to residual sample length
    X0 = X0_full[-n_u:, :]

    # Lagged residuals + trim
    U_lags = lagmat(u, maxlag=nlags, trim="both")
    y = u[nlags:]
    X = np.column_stack([X0[nlags:, :], U_lags])

    if not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError("Non-finite values (NaN/inf) in BG auxiliary regression inputs.")

    aux_res = sm.OLS(y, X).fit()

    n = aux_res.nobs
    lm = n * aux_res.rsquared
    lm_p = stats.chi2.sf(lm, nlags)

    k = X.shape[1]
    R = np.zeros((nlags, k))
    R[:, -nlags:] = np.eye(nlags)
    f_test = aux_res.f_test(R)

    return {
        "LM stat": float(lm),
        "LM p-value": float(lm_p),
        "F stat": float(f_test.fvalue),
        "F p-value": float(f_test.pvalue),
        "nobs_aux": int(n),
        "resid_len": int(n_u),
        "exog_rows_full": int(X0_full.shape[0]),
    }

bg_out = bg_test_ardl_aligned(res, nlags=2)
bg_out


{'LM stat': 1.9028891397782155,
 'LM p-value': 0.3861827523450359,
 'F stat': 0.9176831351963712,
 'F p-value': 0.40488656288283953,
 'nobs_aux': 65,
 'resid_len': 67,
 'exog_rows_full': 71}

In [9]:
def bp_test_ardl(res):
    u = np.asarray(res.resid)
    X0_full = np.asarray(res.model.data.exog)
    X0 = X0_full[-len(u):, :]  # align to residual sample
    X0 = sm.add_constant(X0, has_constant="add")  # force constant

    bp = het_breuschpagan(u, X0)

    return {
        "LM stat": float(bp[0]),
        "LM p-value": float(bp[1]),
        "F stat": float(bp[2]),
        "F p-value": float(bp[3]),
        "k_exog": int(X0.shape[1]),
    }

bp_out = bp_test_ardl(res)
bp_out


{'LM stat': 12.760897606464054,
 'LM p-value': 0.0016943623506556096,
 'F stat': 7.528677750675967,
 'F p-value': 0.0011577268714001228,
 'k_exog': 3}

In [10]:
import joblib
joblib.dump(res, "../data/joblib/ardl_cpi_transport_brent.joblib")

['../data/joblib/ardl_cpi_transport_brent.joblib']